<a href="https://colab.research.google.com/github/matthewpecsok/IS4490_student_course_files/blob/main/IS4490_module8_inclass_demo_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json

In [ ]:
# Colab setup step 1: install zstd, which the Ollama installer expects on Ubuntu.
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (466 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
# Colab setup step 2: install Ollama.
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


# Start Ollama

RESTART ME IF NEEDED

In [ ]:
# Colab setup step 3: start the Ollama server and pull the model.
import subprocess
import time

ollama_server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

pull = subprocess.run(["ollama", "pull", "gemma4:12b"], capture_output=True, text=True)
print(pull.stdout[-1000:])
print(pull.stderr[-1000:])
print("Ollama setup attempted. If the pull failed, check the runtime log and rerun this cell.")


pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 0cb0cd0a5c49: 100% ▕██████████████████▏ 175 MB                         
pulling bb722270d543: 100% ▕██████████████████▏ 7.4 GB                         
pulling 7008a656050b: 100% ▕██████████████████▏ 465 MB                         
pulling b507b9c2f6ca: 100% ▕██████████████████▏   13 B                         
pulling 0d542e0c8804: 100% ▕██████████████████▏  10 KB                         
pulling 2365fbb6d97b: 100% ▕██████████████████▏   64 B                         
pulling e5ca99f91896: 100% ▕██████████████████▏  242 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 

Ollama setup attempted. If the pull failed, check the runtime log and rerun this cell.


In [ ]:
# Colab setup step 4: install Python packages.
!pip install langchain langchain-ollama --quiet

In [ ]:
from datetime import datetime

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

MODEL = "gemma4:12b"
llm = ChatOllama(model=MODEL, temperature=0)

print(f"Ready. Using {MODEL} through Ollama and LangChain.")

Ready. Using gemma4:12b through Ollama and LangChain.


In [ ]:
llm.invoke("what is the capitol of france?")

AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'model': 'gemma4:12b', 'created_at': '2026-10-15T18:41:55.249387205Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24001494148, 'load_duration': 22283100784, 'prompt_eval_count': 23, 'prompt_eval_duration': 450932000, 'eval_count': 55, 'eval_duration': 1264735000, 'logprobs': None, 'model_name': 'gemma4:12b', 'model_provider': 'ollama'}, id='lc_run--01a140de-946e-7b22-afa4-a2bf63e58421-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 23, 'output_tokens': 55, 'total_tokens': 78})

In [ ]:
!mkdir documents
!mkdir law_documents
!mkdir policy_documents

In [ ]:
!echo "This file is a law document." > ./documents/law.txt
!echo "This file is a policy document." > ./documents/policy.txt


In [ ]:
!echo "This file is customer email document. it's neither a policy nor law document" > ./documents/unknown.txt

# Tools

In [ ]:
import os
import shutil
import json
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

# --- TOOLS ---

@tool
def list_files(directory: str = "/content/") -> str:
    """List all files inside the specified directory."""
    try:
        files = os.listdir(directory)
        return json.dumps(files)
    except Exception as e:
        return f"Error listing directory {directory}: {str(e)}"

@tool
def read_file(file_path: str) -> str:
    """Read and return the text contents of the given file path."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        return f"Error reading file {file_path}: {str(e)}"

@tool
def move_file(source_path: str, destination_path: str) -> str:
    """Move a file from source_path to destination_path, creating directories if needed."""
    try:
        dest_dir = os.path.dirname(destination_path)
        shutil.move(source_path, destination_path)
        return f"Successfully moved {source_path} to {destination_path}"
    except Exception as e:
        return f"Error moving {source_path} to {destination_path}: {str(e)}"

# Map tools for dynamic lookups inside the loop
tools_map = {
    "list_files": list_files,
    "read_file": read_file,
    "move_file": move_file
}

# User Instructions

In [ ]:
# Define your custom instructions / goal for the file-organizing agent
user_instructions = """
I need assistance organizing some files
Organize the .txt files found inside '/content/documents/'
by reading their content, deciding on the type of file they are (law,policy)
and moving law files to '/content/law_documents/'
and policy files to '/content/policy_documents/'
finally summarize the work you did for review.
"""

# Agent System Instructions

In [ ]:
# Fallback instruction if not set in the preceding cell
goal = globals().get('user_instructions', "Organize the files in /content/")

print(f"Starting agent loop with goal: '{goal}'")

system_prompt = f"""
You are a file-organizing agent.

Your Goal/Task:
{goal}

You can call these tools:
- list_files (list directory contents, defaults to '/content/')
- read_file (read text file contents)
- move_file (move a file to a new destination)

Guidelines:
1. If you do not know the contents of a directory, call `list_files` first.
2. Run sequentially. If you need to inspect multiple files, read them step by step.
3. Make decisions based only on observations inside your memory.
4. When you are finished executing the user's task, return "action": "FINAL".

You MUST respond with a JSON object in this exact format:
{{
    "action": "list_files" | "read_file" | "move_file" | "FINAL",
    "directory": "/content/",
    "file_path": "/content/file.txt",
    "source_path": "/content/file.txt",
    "destination_path": "/content/documents/file.txt"
}}
"""

Starting agent loop with goal: '
I need assistance organizing some files
Organize the .txt files found inside '/content/documents/' 
by reading their content, deciding on the type of file they are (law,policy)
and moving law files to '/content/law_documents/' 
and policy files to '/content/policy_documents/'
finally summarize the work you did for review.
'


# Agent LOOP

In [ ]:
# --- AGENT LOOP ---

working_directory = "/content/"
agent_memory = []
finished = False
max_iterations = 15
iteration = 0



while not finished and iteration < max_iterations:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")

    # Dynamic context populated solely from memory
    # this injects the history/memory into the prompt
    # so the agent 'remembers' what happened so far
    # and can then decide what to do next
    user_prompt = f"""
    Previous actions and observations:
    {json.dumps(agent_memory, indent=2)}
    Based on your memory of previous steps, decide the next action.
    """

    try:
        # call the llm to make a decision
        response = llm.invoke(
            [
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt)
            ],
            response_format={"type": "json_object"}
        )
        # action data is simply a dictionary
        # that holds the decision made by the LLM
        action_data = json.loads(response.content)
        print(f"Agent Decided: {action_data}")

        action = action_data.get("action")

        # if the agent used FINAL it believes it is done
        if action == "FINAL":
            print("Agent completed its organization goal.")
            response = llm.invoke(
            [
                SystemMessage(content="Summarize what happened."),
                HumanMessage(content=user_prompt)
            ])
            print(response.content)
            finished = True

        # the agent has selected an action/tool
        elif action in tools_map:
            selected_tool = tools_map[action]

            tool_args = {}
            if action == "list_files":
                tool_args = {"directory": action_data.get("directory", working_directory)}
            elif action == "read_file":
                tool_args = {"file_path": action_data.get("file_path")}
            elif action == "move_file":
                tool_args = {
                    "source_path": action_data.get("source_path"),
                    "destination_path": action_data.get("destination_path")
                }

            result = selected_tool.invoke(tool_args)
            agent_memory.append({"action": action, "arguments": tool_args, "result": result})

        # something went wrong here. the agent picked an action that doesn't exist.
        else:
            print(f"Unknown action received: {action}")
            agent_memory.append({"error": f"Unknown action: {action}"})

    except Exception as e:
        print(f"Error in iteration: {e}")
        break


--- Iteration 1 ---
Agent Decided: {'action': 'list_files', 'directory': '/content/documents/'}

--- Iteration 2 ---
Agent Decided: {'action': 'read_file', 'file_path': '/content/documents/law.txt'}

--- Iteration 3 ---
Agent Decided: {'action': 'read_file', 'file_path': '/content/documents/unknown.txt'}

--- Iteration 4 ---
Agent Decided: {'action': 'read_file', 'file_path': '/content/documents/policy.txt'}

--- Iteration 5 ---
Agent Decided: {'action': 'move_file', 'source_path': '/content/documents/law.txt', 'destination_path': '/content/law_documents/law.txt'}

--- Iteration 6 ---
Agent Decided: {'action': 'move_file', 'source_path': '/content/documents/policy.txt', 'destination_path': '/content/policy_documents/policy.txt'}

--- Iteration 7 ---
Agent Decided: {'action': 'FINAL'}
Agent completed its organization goal.
The following actions were performed:

1.  **File Listing:** The contents of the `/content/documents/` directory were listed, revealing three files: `law.txt`, `unkn

In [ ]:
agent_memory

[{'action': 'list_files',
  'arguments': {'directory': '/content/documents/'},
  'result': '["law.txt", "policy.txt"]'},
 {'action': 'read_file',
  'arguments': {'file_path': '/content/documents/law.txt'},
  'result': 'This file is a law document.\n'},
 {'action': 'read_file',
  'arguments': {'file_path': '/content/documents/policy.txt'},
  'result': 'This file is a policy document.\n'},
 {'action': 'move_file',
  'arguments': {'source_path': '/content/documents/law.txt',
   'destination_path': '/content/law_documents/law.txt'},
  'result': 'Successfully moved /content/documents/law.txt to /content/law_documents/law.txt'},
 {'action': 'move_file',
  'arguments': {'source_path': '/content/documents/policy.txt',
   'destination_path': '/content/policy_documents/policy.txt'},
  'result': 'Successfully moved /content/documents/policy.txt to /content/policy_documents/policy.txt'}]